In [ ]:
# Imports
!pip install groq faiss-cpu -q

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from groq import Groq
import os

print("imports successful")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.0 MB/s eta 0:00:00
imports successful


In [ ]:
# Knowledge Agent (Agent 1)

# Medical knowledge base
medical_docs = [
    "Myocardial infarction causes chest pain, left arm numbness and shortness of breath.",
    "Angina pectoris causes chest pain triggered by exertion or stress.",
    "Pulmonary embolism causes sudden chest pain and shortness of breath.",
    "Anxiety disorders can mimic cardiac symptoms including chest tightness.",
    "Aortic dissection causes severe tearing chest pain radiating to the back.",
    "GERD causes burning chest pain that worsens after eating.",
    "Troponin levels rise within 3 to 6 hours of myocardial infarction.",
    "ECG changes in MI include ST elevation and T wave inversion.",
    "Acute coronary syndrome requires immediate aspirin and ECG monitoring.",
    "Pneumonia causes chest pain, fever, and productive cough.",
    "Hypertensive crisis causes severe headache, chest pain and vision changes.",
    "Costochondritis causes sharp chest pain that worsens with touch or movement.",
    "Pericarditis causes sharp chest pain that improves when sitting forward.",
    "Heart failure causes shortness of breath, fatigue and leg swelling.",
    "Stable angina is relieved by rest and nitrates within a few minutes."
]


print("Loading embedding model")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Here converting documents to vectors
print("Indexing documents")
doc_vectors = embedder.encode(medical_docs)
doc_vectors = np.array(doc_vectors).astype('float32')

dimension = doc_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_vectors)
print(f"Knowledge base ready — {len(medical_docs)} documents indexed")
print(f"Vector dimension: {dimension}")

def knowledge_agent(patient_case, k=3):
    query_vector = embedder.encode([patient_case])
    query_vector = np.array(query_vector).astype('float32')
    distances, indices = index.search(query_vector, k)
    results = []
    for i, idx in enumerate(indices[0]):
        score = round(1 / (1 + distances[0][i]), 3)
        results.append({
            "rank": i + 1,
            "document": medical_docs[idx],
            "relevance_score": score
        })
    return results

# Testing Agent 1
patient_case = "45 year old male with chest pain, left arm numbness and shortness of breath"
retrieved = knowledge_agent(patient_case)

print(f"\n Agent 1 — Knowledge Agent Results")
print(f"Patient: {patient_case}\n")
for doc in retrieved:
    print(f"Rank {doc['rank']} — Score: {doc['relevance_score']}")
    print(f"  {doc['document']}\n")

print("Agent 1 working successfully")

Loading embedding model


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexing documents
Knowledge base ready — 15 documents indexed
Vector dimension: 384

 Agent 1 — Knowledge Agent Results
Patient: 45 year old male with chest pain, left arm numbness and shortness of breath

Rank 1 — Score: 0.6679999828338623
  Myocardial infarction causes chest pain, left arm numbness and shortness of breath.

Rank 2 — Score: 0.5740000009536743
  Pulmonary embolism causes sudden chest pain and shortness of breath.

Rank 3 — Score: 0.5410000085830688
  Heart failure causes shortness of breath, fatigue and leg swelling.

Agent 1 working successfully


In [ ]:
# Reasoning Agent (Agent 2)

# Groq API key
os.environ["GROQ_API_KEY"] = "PASTE_YOUR_GROQ_API_KEY_HERE"
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def reasoning_agent(patient_case, retrieved_docs):

    # Here formating retrieved documents
    docs_text = ""
    for doc in retrieved_docs:
        docs_text += f"- {doc['document']} (relevance: {doc['relevance_score']})\n"

    prompt = f"""You are a clinical reasoning assistant.

Patient Case: {patient_case}

Relevant Medical Knowledge:
{docs_text}

Based on the patient case and the medical knowledge provided, please:
1. Reason step by step through the symptoms
2. List your top 3 differential diagnoses ranked by probability
3. For each diagnosis provide a confidence score between 0 and 1
4. Explain which symptoms support each diagnosis

Format your response exactly like this:
REASONING: [your step by step reasoning]
DIAGNOSIS 1: [name] | CONFIDENCE: [score] | EVIDENCE: [supporting symptoms]
DIAGNOSIS 2: [name] | CONFIDENCE: [score] | EVIDENCE: [supporting symptoms]
DIAGNOSIS 3: [name] | CONFIDENCE: [score] | EVIDENCE: [supporting symptoms]"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an expert clinical diagnosis assistant. Always reason step by step."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

# Testing Agent 2
print("Agent 2 — Reasoning Agent")

reasoning_output = reasoning_agent(patient_case, retrieved)
print(reasoning_output)
print("\n Agent 2 working successfully")

Agent 2 — Reasoning Agent
REASONING: The patient presents with chest pain, left arm numbness, and shortness of breath. First, we consider myocardial infarction, which is known to cause chest pain, left arm numbness, and shortness of breath, matching all the patient's symptoms. Next, pulmonary embolism is considered, as it causes sudden chest pain and shortness of breath, aligning with two of the patient's symptoms. Lastly, heart failure, which causes shortness of breath among other symptoms, is also a possibility but seems less likely given the presence of chest pain and left arm numbness, which are not typical primary symptoms of heart failure. The relevance scores provided in the medical knowledge suggest myocardial infarction has the highest relevance to the patient's symptoms, followed by pulmonary embolism, and then heart failure.

DIAGNOSIS 1: Myocardial Infarction | CONFIDENCE: 0.8 | EVIDENCE: Chest pain, left arm numbness, and shortness of breath are all classic symptoms of myo

In [ ]:
# Critique Agent (Agent 3)

def critique_agent(patient_case, reasoning_output, retrieved_docs):

    docs_text = ""
    for doc in retrieved_docs:
        docs_text += f"- {doc['document']}\n"

    prompt = f"""You are an expert medical critic providing a second opinion.

Patient Case: {patient_case}

Primary Reasoning and Diagnoses:
{reasoning_output}

Supporting Medical Knowledge:
{docs_text}

Your job is to critically challenge the above diagnoses by:
1. Identifying any evidence AGAINST the primary diagnosis
2. Suggesting any important diagnoses that may have been MISSED
3. Asking what additional tests would CONFIRM or RULE OUT the primary diagnosis
4. Providing an adjusted final confidence after your critique

Format your response exactly like this:
CRITIQUE: [your challenge to the primary diagnosis]
MISSED_DIAGNOSES: [any important diagnoses not considered]
RECOMMENDED_TESTS: [tests needed to confirm or rule out]
AGREEMENT_SCORE: [0 to 1, how much evidence supports primary diagnosis]
REBUTTAL_SCORE: [0 to 1, how much evidence contradicts primary diagnosis]
FINAL_DIAGNOSIS: [confirmed or modified diagnosis]
FINAL_CONFIDENCE: [adjusted confidence score 0 to 1]"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a senior medical consultant providing adversarial second opinions to improve diagnostic accuracy."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

# Testing Agent 3
print(" Agent 3 — Critique Agent")

critique_output = critique_agent(patient_case, reasoning_output, retrieved)
print(critique_output)
print("\n Agent 3 working!")

 Agent 3 — Critique Agent
CRITIQUE: The primary diagnosis of myocardial infarction is based on the presence of chest pain, left arm numbness, and shortness of breath. However, the evidence against this diagnosis includes the lack of information on the patient's electrocardiogram (ECG) results, troponin levels, and echocardiogram findings, which are crucial for confirming myocardial infarction. Additionally, the presence of left arm numbness can also be associated with other conditions such as musculoskeletal or neurological disorders, which may have been overlooked.

MISSED_DIAGNOSES: Important diagnoses that may have been missed include acute coronary syndrome, cardiac tamponade, and aortic dissection, which can also present with chest pain and shortness of breath. Furthermore, the possibility of pneumothorax, which can cause sudden chest pain and shortness of breath, should also be considered.

RECOMMENDED_TESTS: To confirm or rule out the primary diagnosis of myocardial infarction, 

In [ ]:
# SHAP + Full Pipeline Combined
import shap
import re

# SHAP Function
def compute_shap(patient_case, diagnosis):

    tokens = patient_case.lower().split()

    def predict(texts):
        scores = []
        for text in texts:
            words = text.split()
            overlap = len(set(words) & set(tokens))
            score = overlap / max(len(tokens), 1)
            scores.append([1 - score, score])
        return np.array(scores)

    explainer = shap.Explainer(predict, shap.maskers.Text(r"\s+"))
    shap_values = explainer([patient_case])

    token_scores = {}
    for i, token in enumerate(shap_values[0].data):
        if i < len(shap_values[0].values):
            score = float(np.mean(np.abs(shap_values[0].values[i])))
            token_scores[token] = round(score, 4)

    sorted_scores = sorted(
        token_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_scores[:8]

def extract_final_confidence(critique_text):
    match = re.search(r'FINAL_CONFIDENCE:\s*([\d.]+)', critique_text)
    if match:
        return float(match.group(1))
    return 0.5

def extract_final_diagnosis(critique_text):
    match = re.search(r'FINAL_DIAGNOSIS:\s*(.+)', critique_text)
    if match:
        return match.group(1).strip()
    return "Unknown"

def extract_tests(critique_text):
    match = re.search(r'RECOMMENDED_TESTS:\s*(.+?)(?=\n[A-Z])', critique_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return "Further clinical assessment required"

# Full Pipeline
def run_full_pipeline(patient_case):
    print(" MEDIAGENT — EXPLAINABLE CLINICAL DECISION SUPPORT")
    print(f"\n Patient Case:\n{patient_case}\n")

    # Agent 1
    print(" Running Agent 1 Knowledge Retrieval...")
    retrieved = knowledge_agent(patient_case, k=3)
    print(f"Retrieved {len(retrieved)} documents\n")

    # Agent 2
    print("  Running Agent 2 Clinical Reasoning...")
    reasoning = reasoning_agent(patient_case, retrieved)
    print(" Diagnosis generated\n")

    # Agent 3
    print(" Running Agent 3 Adversarial Critique...")
    critique = critique_agent(patient_case, reasoning, retrieved)
    print(" Critique complete\n")

    # SHAP
    print(" Running SHAP Explanation...")
    final_diagnosis = extract_final_diagnosis(critique)
    shap_scores = compute_shap(patient_case, final_diagnosis)
    print(" Explanation generated\n")

    # Final confidence
    final_confidence = extract_final_confidence(critique)
    recommended_tests = extract_tests(critique)

    # Output
    print(" FINAL OUTPUT")
    print(f"\n Final Diagnosis:  {final_diagnosis}")
    print(f" Confidence Score: {final_confidence}")

    print(f"\n SHAP Explanation — Key Contributing Factors:")
    print(f" (Which words most influenced the diagnosis)")
    for token, score in shap_scores:
        bar = "█" * int(score * 100)
        print(f"   {token:<20} {score:.4f}  {bar}")

    print(f"\n Recommended Next Steps:")
    print(f"   {recommended_tests[:300]}")

    print(f"\n DISCLAIMER: This system is for research purposes only.")
    print(f"   Always consult a qualified clinician for medical decisions.")

    return {
        "patient_case": patient_case,
        "final_diagnosis": final_diagnosis,
        "final_confidence": final_confidence,
        "shap_scores": shap_scores,
        "recommended_tests": recommended_tests,
        "full_reasoning": reasoning,
        "full_critique": critique
    }

# Run the full pipeline
result = run_full_pipeline(patient_case)
print("\n pipeline working successfull")

 MEDIAGENT — EXPLAINABLE CLINICAL DECISION SUPPORT

 Patient Case:
45 year old male with chest pain, left arm numbness and shortness of breath

 Running Agent 1 Knowledge Retrieval...
Retrieved 3 documents

  Running Agent 2 Clinical Reasoning...
 Diagnosis generated

 Running Agent 3 Adversarial Critique...
 Critique complete

 Running SHAP Explanation...
 Explanation generated

 FINAL OUTPUT

 Final Diagnosis:  Myocardial Infarction remains a leading diagnosis but with a need for further diagnostic workup to confirm.
 Confidence Score: 0.65

 SHAP Explanation — Key Contributing Factors:
 (Which words most influenced the diagnosis)
   45                   0.1071  ██████████
   year                 0.0714  ███████
   old                  0.0714  ███████
   male                 0.0714  ███████
   with                 0.0714  ███████
   chest                0.0714  ███████
   pain,                0.0714  ███████
   left                 0.0714  ███████

 Recommended Next Steps:
   To conf

In [ ]:
# Baseline Evaluation

# 10 test cases with correct answers
test_cases = [
    {"case": "Patient has chest pain, left arm numbness and shortness of breath", "answer": "Myocardial Infarction"},
    {"case": "Patient has sudden sharp chest pain and difficulty breathing after long flight", "answer": "Pulmonary Embolism"},
    {"case": "Patient has burning chest pain that worsens after eating and at night", "answer": "GERD"},
    {"case": "Patient has sharp chest pain that improves when sitting forward", "answer": "Pericarditis"},
    {"case": "Patient has chest pain with tenderness when pressing on the chest wall", "answer": "Costochondritis"},
    {"case": "Patient has chest pain triggered by walking uphill, relieved by rest", "answer": "Angina Pectoris"},
    {"case": "Patient has severe tearing chest pain radiating to the back", "answer": "Aortic Dissection"},
    {"case": "Patient has chest pain with fever, cough and productive sputum", "answer": "Pneumonia"},
    {"case": "Patient has shortness of breath, leg swelling and fatigue", "answer": "Heart Failure"},
    {"case": "Patient has chest tightness, palpitations and anxiety with no physical cause", "answer": "Anxiety Disorder"},
]

# Baseline 1 Single agent with no RAG
def baseline1_single_agent(patient_case):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a clinical diagnosis assistant."},
            {"role": "user", "content": f"Patient case: {patient_case}\nGive ONE diagnosis only. Reply with just the diagnosis name."}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()

# Baseline 2 with RAG plus Single agent
def baseline2_rag_single(patient_case):
    docs = knowledge_agent(patient_case, k=3)
    docs_text = " ".join([d["document"] for d in docs])
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a clinical diagnosis assistant."},
            {"role": "user", "content": f"Patient: {patient_case}\nKnowledge: {docs_text}\nGive ONE diagnosis only. Reply with just the diagnosis name."}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()

# Proposed system top diagnosis
def proposed_system_diagnosis(patient_case):
    docs = knowledge_agent(patient_case, k=3)
    reasoning = reasoning_agent(patient_case, docs)
    # Extracting first diagnosis here
    import re
    match = re.search(r'DIAGNOSIS 1:\s*([^|]+)', reasoning)
    if match:
        return match.group(1).strip()
    return "Unknown"

# Checking if answer is correct
def is_correct(prediction, answer):
    return answer.lower() in prediction.lower() or prediction.lower() in answer.lower()

# Running evaluation
print(" Running Baseline Evaluation")

b1_correct = 0
b2_correct = 0
proposed_correct = 0

results_table = []

for i, test in enumerate(test_cases):
    case = test["case"]
    answer = test["answer"]

    print(f"\nTest {i+1}: {answer}")

    b1 = baseline1_single_agent(case)
    b2 = baseline2_rag_single(case)
    prop = proposed_system_diagnosis(case)

    b1_ok = is_correct(b1, answer)
    b2_ok = is_correct(b2, answer)
    prop_ok = is_correct(prop, answer)

    b1_correct += b1_ok
    b2_correct += b2_ok
    proposed_correct += prop_ok

    print(f"  Baseline 1: {b1[:40]} → {'' if b1_ok else ''}")
    print(f"  Baseline 2: {b2[:40]} → {'' if b2_ok else ''}")
    print(f"  Proposed:   {prop[:40]} → {'' if prop_ok else ''}")

    results_table.append({
        "case": answer,
        "b1": "" if b1_ok else "",
        "b2": "" if b2_ok else "",
        "proposed": "" if prop_ok else ""
    })

# results
print(" EVALUATION RESULTS SUMMARY")
print(f"\n{'System':<30} {'Correct':<10} {'Accuracy'}")
print(f"{'Baseline 1 (Single Agent)':<30} {b1_correct}/10      {b1_correct*10}%")
print(f"{'Baseline 2 (RAG + Single)':<30} {b2_correct}/10      {b2_correct*10}%")
print(f"{'Proposed System (3 Agents)':<30} {proposed_correct}/10      {proposed_correct*10}%")
print("\n Evaluation complete!")
print("\nFull evaluation on 100 MedQA questions will be run in the next phase.")

 Running Baseline Evaluation

Test 1: Myocardial Infarction
  Baseline 1: Myocardial Infarction → 
  Baseline 2: Myocardial infarction → 
  Proposed:   Myocardial Infarction → 

Test 2: Pulmonary Embolism
  Baseline 1: Pulmonary Embolism → 
  Baseline 2: Pulmonary embolism → 
  Proposed:   Pulmonary Embolism → 

Test 3: GERD
  Baseline 1: Gastroesophageal Reflux Disease (GERD) → 
  Baseline 2: GERD → 
  Proposed:   GERD → 

Test 4: Pericarditis
  Baseline 1: Acute Pericarditis → 
  Baseline 2: Pericarditis → 
  Proposed:   Pericarditis → 

Test 5: Costochondritis
  Baseline 1: Costochondritis → 
  Baseline 2: Costochondritis → 
  Proposed:   Costochondritis → 

Test 6: Angina Pectoris
  Baseline 1: Angina Pectoris → 
  Baseline 2: Angina pectoris → 
  Proposed:   Angina pectoris → 

Test 7: Aortic Dissection
  Baseline 1: Aortic Dissection → 
  Baseline 2: Aortic dissection → 
  Proposed:   Aortic dissection → 

Test 8: Pneumonia
  Baseline 1: Pneumonia → 
  Baseline 2: Pneumonia → 
  

In [ ]:
# MedQA Fixed (Hugging Face direct)
from datasets import load_dataset

print("Downloading MedQA dataset...")
dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")

print(f" Downloaded {len(dataset)} test questions")
print(f"\nSample question:")
sample = dataset[0]
print(f"Q: {sample['question'][:200]}")
print(f"Options: {sample['options']}")
print(f"Answer: {sample['answer']}")

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl: reconstructing file:   0%|          |  0.00B / 16.2MB            

phrases_no_exclude_train.jsonl: downloading bytes:           |  0.00B            

phrases_no_exclude_test.jsonl:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

 Downloaded 1273 test questions

Sample question:
Q: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. Th
Options: {'A': 'Disclose the error to the patient and put it in the operative report', 'B': 'Tell the attending that he cannot fail to disclose this mistake', 'C': 'Report the physician to the ethics committee', 'D': 'Refuse to dictate the operative report'}
Answer: Tell the attending that he cannot fail to disclose this mistake


In [ ]:
# Evaluating on 100 Real MedQA Questions
import time
import re

def call_with_retry(fn, *args, max_retries=6):
    for attempt in range(max_retries):
        try:
            return fn(*args)
        except Exception as e:
            if '429' in str(e):
                wait = (attempt + 1) * 20
                print(f"   ⏳ Rate limit — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise e
    return None

def extract_answer_letter(text, options):
    text = text.strip().upper()
    for letter in ['A', 'B', 'C', 'D']:
        if text.startswith(letter):
            return letter
    for letter, value in options.items():
        if value.lower() in text.lower():
            return letter
    return "A"

def get_correct_letter(answer, options):
    for letter, value in options.items():
        if value.lower() == answer.lower():
            return letter
    return "A"

def baseline1_medqa(question, options):
    opts = "\n".join([f"{k}: {v}" for k, v in options.items()])
    response = client.chat.completions.create(
        # model="llama-3.1-8b-instant",
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a medical expert. Answer with only the letter A, B, C, or D."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts}\n\nAnswer with only the letter:"}
        ],
        temperature=0.1,
        max_tokens=5
    )
    return extract_answer_letter(response.choices[0].message.content, options)

def proposed_medqa(question, options):
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    docs = knowledge_agent(question, k=3)
    docs_text = "\n".join([f"- {d['document']}" for d in docs])
    response = client.chat.completions.create(
        # model="llama-3.1-8b-instant",
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a clinical reasoning expert. Answer with only the letter A, B, C, or D."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts_text}\n\nRelevant Knowledge:\n{docs_text}\n\nReason step by step then answer with only the letter:"}
        ],
        temperature=0.1,
        max_tokens=5
    )
    return extract_answer_letter(response.choices[0].message.content, options)

print("Evaluating Baseline 1 (no RAG) vs Proposed System (RAG) on 100 MedQA questions")

b1_correct = 0
prop_correct = 0

for i in range(100):
    sample = dataset[i]
    question = sample['question']
    options = sample['options']
    correct_letter = get_correct_letter(sample['answer'], options)

    b1_ans = call_with_retry(baseline1_medqa, question, options)
    time.sleep(1.2)
    prop_ans = call_with_retry(proposed_medqa, question, options)
    time.sleep(1.2)

    if b1_ans == correct_letter:
        b1_correct += 1
    if prop_ans == correct_letter:
        prop_correct += 1

    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/100 | Baseline 1: {b1_correct}/{i+1} | Proposed: {prop_correct}/{i+1}")

print("TABLE 2 — MedQA-USMLE BENCHMARK (100 Questions)")
print(f"Baseline 1 (Single-Agent LLM, no RAG):  {b1_correct}/100  ({b1_correct}%)")
print(f"Proposed System (RAG-Augmented):        {prop_correct}/100  ({prop_correct}%)")

import json
with open("medqa_table2_results.json", "w") as f:
    json.dump({
        "baseline_1_no_rag": {"correct": b1_correct, "total": 100, "accuracy": f"{b1_correct}%"},
        "proposed_rag": {"correct": prop_correct, "total": 100, "accuracy": f"{prop_correct}%"}
    }, f, indent=2)

print("\nTable 2 evaluation complete and results are saved to medqa_table2_results.json")

Evaluating Baseline 1 (no RAG) vs Proposed System (RAG) on 100 MedQA questions
Progress: 10/100 | Baseline 1: 6/10 | Proposed: 1/10
Progress: 20/100 | Baseline 1: 14/20 | Proposed: 1/20
Progress: 30/100 | Baseline 1: 22/30 | Proposed: 4/30
Progress: 40/100 | Baseline 1: 30/40 | Proposed: 8/40
Progress: 50/100 | Baseline 1: 37/50 | Proposed: 11/50
Progress: 60/100 | Baseline 1: 45/60 | Proposed: 15/60
Progress: 70/100 | Baseline 1: 51/70 | Proposed: 19/70
Progress: 80/100 | Baseline 1: 60/80 | Proposed: 22/80
Progress: 90/100 | Baseline 1: 69/90 | Proposed: 24/90
Progress: 100/100 | Baseline 1: 78/100 | Proposed: 25/100
TABLE 2 — MedQA-USMLE BENCHMARK (100 Questions)
Baseline 1 (Single-Agent LLM, no RAG):  78/100  (78%)
Proposed System (RAG-Augmented):        25/100  (25%)

Table 2 evaluation complete and results are saved to medqa_table2_results.json


In [ ]:
# Evaluating Full System (with Critique) on same 100 questions
import time, re

def call_with_retry(fn, *args, max_retries=6):
    for attempt in range(max_retries):
        try:
            return fn(*args)
        except Exception as e:
            if '429' in str(e):
                wait = (attempt + 1) * 20
                print(f"   ⏳ Rate limit — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise e
    return None

def baseline3_medqa(question, options):
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    docs = knowledge_agent(question, k=3)
    docs_text = "\n".join([f"- {d['document']}" for d in docs])

    reasoning_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a clinical reasoning expert. Always start your response with 'ANSWER: <letter>' then briefly explain."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts_text}\n\nRelevant Knowledge:\n{docs_text}\n\nStart with 'ANSWER: <letter>', then briefly explain your reasoning in 1-2 sentences."}
        ],
        temperature=0.1,
        max_tokens=120
    )
    initial_reasoning = reasoning_response.choices[0].message.content
    initial_match = re.search(r'ANSWER:\s*([A-D])', initial_reasoning, re.IGNORECASE)
    initial_answer = initial_match.group(1).upper() if initial_match else "A"

    critique_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a critical second reviewer. Always start your response with 'FINAL ANSWER: <letter>' then briefly explain your critique."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts_text}\n\nInitial reasoning: {initial_reasoning}\n\nStart with 'FINAL ANSWER: <letter>' (confirm or correct the initial answer), then briefly explain why in 1-2 sentences."}
        ],
        temperature=0.1,
        max_tokens=120
    )
    critique_text = critique_response.choices[0].message.content
    final_match = re.search(r'FINAL ANSWER:\s*([A-D])', critique_text, re.IGNORECASE)
    final_answer = final_match.group(1).upper() if final_match else initial_answer

    changed = (final_answer != initial_answer)
    return initial_answer, final_answer, changed

print("Evaluating Full System (Knowledge + Reasoning + Critique) on 100 questions")

full_system_correct = 0
critique_changed_count = 0
critique_changed_to_correct = 0
critique_changed_to_wrong = 0

for i in range(100):
    sample = dataset[i]
    question = sample['question']
    options = sample['options']
    correct_letter = get_correct_letter(sample['answer'], options)

    result = call_with_retry(baseline3_medqa, question, options)
    if result is None:
        print(f"Q{i+1}: failed after retries, skipping")
        continue
    initial_ans, final_ans, changed = result
    final_ok = (final_ans == correct_letter)
    initial_ok = (initial_ans == correct_letter)

    full_system_correct += final_ok

    if changed:
        critique_changed_count += 1
        if final_ok and not initial_ok:
            critique_changed_to_correct += 1
        elif not final_ok and initial_ok:
            critique_changed_to_wrong += 1

    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/100 | Full System correct: {full_system_correct}/{i+1}")
    time.sleep(1.5)

print(" FULL SYSTEM (WITH CRITIQUE) — 100 MedQA Questions")
print(f"Full System (Knowledge+Reasoning+Critique): {full_system_correct}/100  ({full_system_correct}%)")
print(f"\nCritique Agent changed the answer on: {critique_changed_count}/100 questions")
print(f" Changed WRONG to CORRECT: {critique_changed_to_correct} times")
print(f" Changed CORRECT to WRONG: {critique_changed_to_wrong} times")
print("\n Full system evaluation complete!")

Evaluating Full System (Knowledge + Reasoning + Critique) on 100 questions
Progress: 10/100 | Full System correct: 6/10
Progress: 20/100 | Full System correct: 12/20
Progress: 30/100 | Full System correct: 19/30
Progress: 40/100 | Full System correct: 25/40
Progress: 50/100 | Full System correct: 31/50
Progress: 60/100 | Full System correct: 40/60
Progress: 70/100 | Full System correct: 47/70
Progress: 80/100 | Full System correct: 54/80
Progress: 90/100 | Full System correct: 63/90
Progress: 100/100 | Full System correct: 72/100
 FULL SYSTEM (WITH CRITIQUE) — 100 MedQA Questions
Full System (Knowledge+Reasoning+Critique): 72/100  (72%)

Critique Agent changed the answer on: 0/100 questions
 Changed WRONG to CORRECT: 0 times
 Changed CORRECT to WRONG: 0 times

 Full system evaluation complete!


In [ ]:
# Conditions A, B, C and D already done: 73/100
import time, re, json

def call_with_retry(fn, *args, max_retries=6):
    for attempt in range(max_retries):
        try:
            return fn(*args)
        except Exception as e:
            if '429' in str(e):
                wait = (attempt + 1) * 20
                print(f"   ⏳ Rate limit — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise e
    return None

def extract_letter(text):
    m = re.search(r'ANSWER:\s*([A-D])', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    text = text.strip().upper()
    for l in ['A','B','C','D']:
        if text.startswith(l): return l
    return "A"

def cond_A(question, options):
    opts = "\n".join([f"{k}: {v}" for k, v in options.items()])
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a medical expert. Always start your response with 'ANSWER: <letter>'."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts}\n\nStart with 'ANSWER: <letter>', then briefly explain in 1 sentence."}
        ],
        temperature=0.1, max_tokens=60
    )
    return extract_letter(response.choices[0].message.content)

def cond_B(question, options):
    opts = "\n".join([f"{k}: {v}" for k, v in options.items()])
    docs = knowledge_agent(question, k=3)
    docs_text = "\n".join([f"- {d['document']}" for d in docs])
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a clinical reasoning expert. Always start your response with 'ANSWER: <letter>'."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts}\n\nKnowledge:\n{docs_text}\n\nStart with 'ANSWER: <letter>', then briefly explain in 1 sentence."}
        ],
        temperature=0.1, max_tokens=80
    )
    return extract_letter(response.choices[0].message.content)

def cond_C(question, options):
    opts = "\n".join([f"{k}: {v}" for k, v in options.items()])
    reasoning_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a medical expert. Always start your response with 'ANSWER: <letter>' then briefly explain."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts}\n\nStart with 'ANSWER: <letter>', then briefly explain your reasoning in 1-2 sentences."}
        ],
        temperature=0.1, max_tokens=120
    )
    initial_reasoning = reasoning_response.choices[0].message.content
    initial_answer = extract_letter(initial_reasoning)

    critique_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a critical second reviewer. Always start your response with 'FINAL ANSWER: <letter>' then briefly explain your critique."},
            {"role": "user", "content": f"Question: {question}\n\nOptions:\n{opts}\n\nInitial reasoning: {initial_reasoning}\n\nStart with 'FINAL ANSWER: <letter>' (confirm or correct), then briefly explain why in 1-2 sentences."}
        ],
        temperature=0.1, max_tokens=120
    )
    critique_text = critique_response.choices[0].message.content
    m = re.search(r'FINAL ANSWER:\s*([A-D])', critique_text, re.IGNORECASE)
    final_answer = m.group(1).upper() if m else initial_answer
    return final_answer, initial_answer

print("Running Conditions A, B, C on the same 100 questions...")

per_question = []

for i in range(100):
    sample = dataset[i]
    q, opts = sample['question'], sample['options']
    correct = get_correct_letter(sample['answer'], opts)

    a = call_with_retry(cond_A, q, opts)
    time.sleep(1)
    b = call_with_retry(cond_B, q, opts)
    time.sleep(1)
    c_result = call_with_retry(cond_C, q, opts)
    c_final, c_initial = c_result if c_result else ("A", "A")
    time.sleep(1)

    per_question.append({
        "qid": i,
        "correct": correct,
        "A": a, "A_ok": int(a == correct),
        "B": b, "B_ok": int(b == correct),
        "C_final": c_final, "C_initial": c_initial,
        "C_ok": int(c_final == correct), "C_changed": int(c_final != c_initial),
    })

    if (i+1) % 10 == 0:
        a_sum = sum(x['A_ok'] for x in per_question)
        b_sum = sum(x['B_ok'] for x in per_question)
        c_sum = sum(x['C_ok'] for x in per_question)
        print(f"{i+1}/100 | A:{a_sum} B:{b_sum} C:{c_sum}")

with open("conditions_ABC_results.json", "w") as f:
    json.dump(per_question, f, indent=2)

a_sum = sum(x['A_ok'] for x in per_question)
b_sum = sum(x['B_ok'] for x in per_question)
c_sum = sum(x['C_ok'] for x in per_question)
c_changed = sum(x['C_changed'] for x in per_question)

print("FULL ABLATION SUMMARY — All 4 Conditions (llama-3.1-8b-instant)")
print(f"A LLM only (no RAG, no critique):    {a_sum}/100")
print(f"B LLM + RAG (no critique):             {b_sum}/100")
print(f"C LLM + critique (no RAG):             {c_sum}/100  | Critique changed: {c_changed}/100")
print(f"D Full system (RAG + critique):        73/100  | Critique changed: 0/100  [from previous run]")
print("\n Full ablation complete!")

Running Conditions A, B, C on the same 100 questions...
10/100 | A:5 B:6 C:6
20/100 | A:11 B:12 C:12
30/100 | A:19 B:18 C:20
40/100 | A:25 B:23 C:25
50/100 | A:29 B:28 C:29
60/100 | A:38 B:37 C:38
70/100 | A:45 B:43 C:45
80/100 | A:53 B:50 C:53
90/100 | A:62 B:59 C:62
100/100 | A:71 B:68 C:71
FULL ABLATION SUMMARY — All 4 Conditions (llama-3.1-8b-instant)
A LLM only (no RAG, no critique):    71/100
B LLM + RAG (no critique):             68/100
C LLM + critique (no RAG):             71/100  | Critique changed: 1/100
D Full system (RAG + critique):        73/100  | Critique changed: 0/100  [from previous run]

 Full ablation complete!


In [ ]:
import json
with open("conditions_ABC_results.json") as f:
    data = json.load(f)

changed_to_correct = sum(1 for x in data if x['C_changed'] and x['C_ok'] and x['correct'] != x['C_initial'])
changed_to_wrong = sum(1 for x in data if x['C_changed'] and not x['C_ok'])
print(f"Condition C — Changed wrong→correct: {changed_to_correct}")
print(f"Condition C — Changed correct→wrong: {changed_to_wrong}")

Condition C — Changed wrong→correct: 0
Condition C — Changed correct→wrong: 1


In [ ]:
# STATISTICAL EVALUATION Macro F1, McNemar's Test, Confidence Intervals
# Using saved JSON files, no new API calls
import json
import numpy as np
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar

# Loading saved data
with open("conditions_ABC_results.json") as f:
    abc_data = json.load(f)

# Reconstructing arrays
correct_labels = [x['correct'] for x in abc_data]
pred_A = [x['A'] for x in abc_data]
pred_B = [x['B'] for x in abc_data]
pred_C = [x['C_final'] for x in abc_data]

# Macro F1 for each condition
f1_A = f1_score(correct_labels, pred_A, average='macro', zero_division=0)
f1_B = f1_score(correct_labels, pred_B, average='macro', zero_division=0)
f1_C = f1_score(correct_labels, pred_C, average='macro', zero_division=0)

print("Macro F1 Scores (Ablation Conditions A, B, C)")
print(f"Condition A (LLM only):        {f1_A:.3f}")
print(f"Condition B (LLM + RAG):       {f1_B:.3f}")
print(f"Condition C (LLM + Critique):  {f1_C:.3f}")

# Bootstraping 95% Confidence Intervals for Accuracy
def bootstrap_ci(correct_flags, n_bootstrap=2000, ci=95):
    correct_flags = np.array(correct_flags)
    n = len(correct_flags)
    boot_accs = []
    for _ in range(n_bootstrap):
        sample_idx = np.random.choice(n, n, replace=True)
        boot_accs.append(correct_flags[sample_idx].mean())
    lower = np.percentile(boot_accs, (100-ci)/2)
    upper = np.percentile(boot_accs, 100-(100-ci)/2)
    return correct_flags.mean(), lower, upper

A_ok = [x['A_ok'] for x in abc_data]
B_ok = [x['B_ok'] for x in abc_data]
C_ok = [x['C_ok'] for x in abc_data]

print("\nBootstrap 95% Confidence Intervals (Accuracy)")
for name, flags in [("A", A_ok), ("B", B_ok), ("C", C_ok)]:
    mean, lo, hi = bootstrap_ci(flags)
    print(f"Condition {name}: {mean:.1%}  (95% CI: {lo:.1%} - {hi:.1%})")

# McNemar's Test — Paired comparisons on same questions
def run_mcnemar(cond1_ok, cond2_ok, name1, name2):
    both_correct = sum(1 for a,b in zip(cond1_ok,cond2_ok) if a==1 and b==1)
    only1 = sum(1 for a,b in zip(cond1_ok,cond2_ok) if a==1 and b==0)
    only2 = sum(1 for a,b in zip(cond1_ok,cond2_ok) if a==0 and b==1)
    both_wrong = sum(1 for a,b in zip(cond1_ok,cond2_ok) if a==0 and b==0)
    table = [[both_correct, only1],[only2, both_wrong]]
    result = mcnemar(table, exact=True)
    print(f"{name1} vs {name2}: statistic={result.statistic:.2f}, p-value={result.pvalue:.4f}"
          f"  {'(significant, p<0.05)' if result.pvalue < 0.05 else '(not significant)'}")

print("\n McNemar's Test (Paired Comparisons)")
run_mcnemar(A_ok, B_ok, "Condition A (LLM only)", "Condition B (LLM+RAG)")
run_mcnemar(A_ok, C_ok, "Condition A (LLM only)", "Condition C (LLM+Critique)")

print("\n Statistical analysis complete!")

Macro F1 Scores (Ablation Conditions A, B, C)
Condition A (LLM only):        0.709
Condition B (LLM + RAG):       0.681
Condition C (LLM + Critique):  0.707

Bootstrap 95% Confidence Intervals (Accuracy)
Condition A: 71.0%  (95% CI: 62.0% - 79.0%)
Condition B: 68.0%  (95% CI: 59.0% - 77.0%)
Condition C: 71.0%  (95% CI: 62.0% - 80.0%)

 McNemar's Test (Paired Comparisons)
Condition A (LLM only) vs Condition B (LLM+RAG): statistic=2.00, p-value=0.4531  (not significant)
Condition A (LLM only) vs Condition C (LLM+Critique): statistic=2.00, p-value=1.0000  (not significant)

 Statistical analysis complete!


In [ ]:
# Full + LangGraph setup
!pip install langgraph langchain-core groq faiss-cpu sentence-transformers shap lime -q

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from groq import Groq
import os, re, json
from typing import TypedDict, List, Dict

# Groq client
os.environ["GROQ_API_KEY"] = "PASTE_YOUR_GROQ_API_KEY_HERE"
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Knowledge base
medical_docs = [
    "Myocardial infarction causes chest pain, left arm numbness and shortness of breath.",
    "Angina pectoris causes chest pain triggered by exertion or stress.",
    "Pulmonary embolism causes sudden chest pain and shortness of breath.",
    "Anxiety disorders can mimic cardiac symptoms including chest tightness.",
    "Aortic dissection causes severe tearing chest pain radiating to the back.",
    "GERD causes burning chest pain that worsens after eating.",
    "Troponin levels rise within 3 to 6 hours of myocardial infarction.",
    "ECG changes in MI include ST elevation and T wave inversion.",
    "Acute coronary syndrome requires immediate aspirin and ECG monitoring.",
    "Pneumonia causes chest pain, fever, and productive cough.",
    "Hypertensive crisis causes severe headache, chest pain and vision changes.",
    "Costochondritis causes sharp chest pain that worsens with touch or movement.",
    "Pericarditis causes sharp chest pain that improves when sitting forward.",
    "Heart failure causes shortness of breath, fatigue and leg swelling.",
    "Stable angina is relieved by rest and nitrates within a few minutes."
]

print("Loading embedder...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_vectors = embedder.encode(medical_docs)
doc_vectors = np.array(doc_vectors).astype('float32')
dimension = doc_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_vectors)
print("Knowledge base is ready")

# LangGraph State Definition
class AgentState(TypedDict):
    patient_case: str
    retrieved_docs: List[Dict]
    reasoning_output: str
    shap_explanation: List
    critique_output: str
    lime_explanation: List
    final_diagnosis: str
    final_confidence: float
    recommended_tests: str

# Node 1 Knowledge Agent
def knowledge_node(state: AgentState) -> AgentState:
    query_vector = embedder.encode([state["patient_case"]])
    query_vector = np.array(query_vector).astype('float32')
    distances, indices = index.search(query_vector, 3)
    results = []
    for i, idx in enumerate(indices[0]):
        score = round(1 / (1 + distances[0][i]), 3)
        results.append({"document": medical_docs[idx], "relevance_score": score})
    state["retrieved_docs"] = results
    return state

# Node 2 Reasoning Agent
def reasoning_node(state: AgentState) -> AgentState:
    docs_text = "\n".join([f"- {d['document']} (relevance: {d['relevance_score']})" for d in state["retrieved_docs"]])
    prompt = f"""You are a clinical reasoning assistant.

Patient Case: {state['patient_case']}

Relevant Medical Knowledge:
{docs_text}

Reason step by step, then list your top 3 differential diagnoses ranked by probability with confidence scores.

Format:
REASONING: [reasoning]
DIAGNOSIS 1: [name] | CONFIDENCE: [score] | EVIDENCE: [evidence]
DIAGNOSIS 2: [name] | CONFIDENCE: [score] | EVIDENCE: [evidence]
DIAGNOSIS 3: [name] | CONFIDENCE: [score] | EVIDENCE: [evidence]"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an expert clinical diagnosis assistant. Always reason step by step."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    state["reasoning_output"] = response.choices[0].message.content
    return state

# Node 3 SHAP Explanation
import shap

def shap_node(state: AgentState) -> AgentState:
    case_text = state["patient_case"]
    tokens = case_text.lower().split()

    def predict(texts):
        scores = []
        for text in texts:
            words = text.split()
            overlap = len(set(words) & set(tokens))
            score = overlap / max(len(tokens), 1)
            scores.append([1 - score, score])
        return np.array(scores)

    explainer = shap.Explainer(predict, shap.maskers.Text(r"\s+"))
    shap_values = explainer([case_text])

    token_scores = {}
    for i, token in enumerate(shap_values[0].data):
        if i < len(shap_values[0].values):
            score = float(np.mean(np.abs(shap_values[0].values[i])))
            token_scores[token] = round(score, 4)

    sorted_scores = sorted(token_scores.items(), key=lambda x: x[1], reverse=True)
    state["shap_explanation"] = sorted_scores[:8]
    return state

# Node 4 Critique Agent
def critique_node(state: AgentState) -> AgentState:
    docs_text = "\n".join([f"- {d['document']}" for d in state["retrieved_docs"]])
    prompt = f"""You are an expert medical critic providing a second opinion.

Patient Case: {state['patient_case']}

Primary Reasoning and Diagnoses:
{state['reasoning_output']}

Supporting Medical Knowledge:
{docs_text}

Challenge the primary diagnosis: identify counter-evidence, missed diagnoses, recommended tests, and an adjusted confidence.

Format:
CRITIQUE: [challenge]
MISSED_DIAGNOSES: [missed]
RECOMMENDED_TESTS: [tests]
AGREEMENT_SCORE: [0 to 1]
REBUTTAL_SCORE: [0 to 1]
FINAL_DIAGNOSIS: [diagnosis]
FINAL_CONFIDENCE: [0 to 1]"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a senior medical consultant providing adversarial second opinions."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    critique_text = response.choices[0].message.content
    state["critique_output"] = critique_text

    match_conf = re.search(r'FINAL_CONFIDENCE:\s*([\d.]+)', critique_text)
    state["final_confidence"] = float(match_conf.group(1)) if match_conf else 0.5

    match_diag = re.search(r'FINAL_DIAGNOSIS:\s*(.+)', critique_text)
    state["final_diagnosis"] = match_diag.group(1).strip() if match_diag else "Unknown"

    match_tests = re.search(r'RECOMMENDED_TESTS:\s*(.+?)(?=\n[A-Z])', critique_text, re.DOTALL)
    state["recommended_tests"] = match_tests.group(1).strip() if match_tests else "Further assessment required"

    return state

print("All LangGraph nodes defined")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.1 MB/s eta 0:00:00
Loading embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base is ready
All LangGraph nodes defined


In [ ]:
# LIME + Urgency + Patient Summary
from lime.lime_text import LimeTextExplainer
from langgraph.graph import StateGraph, END

# Final AgentState with all fields
class AgentState(TypedDict):
    patient_case: str
    retrieved_docs: List[Dict]
    reasoning_output: str
    shap_explanation: List
    critique_output: str
    lime_explanation: List
    final_diagnosis: str
    final_confidence: float
    recommended_tests: str
    patient_summary: str
    urgency_level: str
    urgency_reason: str

# LIME Node
def lime_node(state: AgentState) -> AgentState:
    case_text = state["patient_case"]
    final_diagnosis = state["final_diagnosis"]
    diagnosis_keywords = set(final_diagnosis.lower().split())

    def predict_proba(texts):
        results = []
        for text in texts:
            words = set(text.lower().split())
            overlap = len(words & diagnosis_keywords)
            relevance_score = min(overlap / max(len(diagnosis_keywords), 1), 1.0)
            results.append([1 - relevance_score, relevance_score])
        return np.array(results)

    explainer = LimeTextExplainer(class_names=["Not Relevant", "Relevant"])
    try:
        exp = explainer.explain_instance(case_text, predict_proba, num_features=8, num_samples=200)
        lime_scores = exp.as_list()
    except Exception:
        lime_scores = [("error", 0.0)]

    state["lime_explanation"] = lime_scores
    return state

# Urgency Node
def urgency_node(state: AgentState) -> AgentState:
    prompt = f"""Based on the following clinical assessment, classify the urgency
of this case into exactly ONE of these three tiers:

- EMERGENCY: Life-threatening, needs immediate emergency care (call ambulance / go to A&E now)
- URGENT: Needs medical attention within 24-48 hours (same-day or next-day GP/urgent care)
- ROUTINE: Can be monitored, book a routine GP appointment if it persists or worsens

Patient case: {state['patient_case']}
Diagnosis under consideration: {state['final_diagnosis']}
Clinical reasoning: {state['reasoning_output']}
Critique notes: {state['critique_output']}

Respond in exactly this format:
URGENCY: [EMERGENCY / URGENT / ROUTINE]
REASON: [1-2 sentences explaining why, in plain language]"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a triage assistant. You classify urgency only. You NEVER suggest medications, drug names, or dosages under any circumstances."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    text = response.choices[0].message.content

    match_level = re.search(r'URGENCY:\s*(EMERGENCY|URGENT|ROUTINE)', text, re.IGNORECASE)
    state["urgency_level"] = match_level.group(1).upper() if match_level else "ROUTINE"

    match_reason = re.search(r'REASON:\s*(.+)', text, re.DOTALL)
    state["urgency_reason"] = match_reason.group(1).strip() if match_reason else "Please consult a healthcare professional to assess urgency."

    return state

# Patient Summary Node
def patient_summary_node(state: AgentState) -> AgentState:
    prompt = f"""You are explaining a medical AI assessment to a patient with no medical training.

CRITICAL RULE: Do NOT mention any medication names, drug types, or dosages,
even common over-the-counter ones. Only mention tests, urgency level, and
whether/when to see a doctor.

CRITICAL FORMATTING RULE: You MUST output your response as exactly 3 numbered
points, each on its own line, using the format "1. ...", "2. ...", "3. ...".
Do NOT merge them into a paragraph. Each point should be 1-2 short sentences.

Write exactly 3 numbered points:
1. What we think might be going on (plain language, no jargon)
2. Why we think this (reference their actual symptoms specifically)
3. What to do next (when to see a doctor — NOT what medication to take)

Patient's symptoms: {state['patient_case']}
Diagnosis: {state['final_diagnosis']}
Reasoning: {state['reasoning_output']}
Critique: {state['critique_output']}
Tests: {state['recommended_tests']}
Urgency: {state['urgency_level']} — {state['urgency_reason']}

Respond with ONLY the 3 numbered points, nothing before or after."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a compassionate medical communicator. You NEVER suggest medications or dosages under any circumstances. You ALWAYS format your response as exactly 3 numbered points on separate lines, never as a single paragraph."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    state["patient_summary"] = response.choices[0].message.content
    return state

print("LIME, Urgency, and Patient Summary nodes defined")

# Build the FINAL graph
workflow = StateGraph(AgentState)
workflow.add_node("knowledge_agent", knowledge_node)
workflow.add_node("reasoning_agent", reasoning_node)
workflow.add_node("shap_explainer", shap_node)
workflow.add_node("critique_agent", critique_node)
workflow.add_node("lime_explainer", lime_node)
workflow.add_node("urgency_classifier", urgency_node)
workflow.add_node("patient_summary", patient_summary_node)

workflow.set_entry_point("knowledge_agent")
workflow.add_edge("knowledge_agent", "reasoning_agent")
workflow.add_edge("reasoning_agent", "shap_explainer")
workflow.add_edge("shap_explainer", "critique_agent")
workflow.add_edge("critique_agent", "lime_explainer")
workflow.add_edge("lime_explainer", "urgency_classifier")
workflow.add_edge("urgency_classifier", "patient_summary")
workflow.add_edge("patient_summary", END)

clinical_graph_v2 = workflow.compile()
print("Final graph built:")
print("knowledge → reasoning → SHAP → critique → LIME → urgency → patient_summary → END")
print(f"\nNodes in graph: {list(workflow.nodes.keys())}")

# Testing
test_case = "45 year old male with chest pain, left arm numbness and shortness of breath"
initial_state = {
    "patient_case": test_case, "retrieved_docs": [], "reasoning_output": "",
    "shap_explanation": [], "critique_output": "", "lime_explanation": [],
    "final_diagnosis": "", "final_confidence": 0.0, "recommended_tests": "",
    "patient_summary": "", "urgency_level": "", "urgency_reason": ""
}

final_state_v2 = clinical_graph_v2.invoke(initial_state)

print(f"\ Urgency: {final_state_v2['urgency_level']}")
print(f"  Summary: {final_state_v2['patient_summary']}")
print("\n Full pipeline working — LIME, urgency, and patient summary all functioning!")

<>:155: SyntaxWarning: invalid escape sequence '\ '
<>:155: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_4627/1491000185.py:155: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ Urgency: {final_state_v2['urgency_level']}")


LIME, Urgency, and Patient Summary nodes defined
Final graph built:
knowledge → reasoning → SHAP → critique → LIME → urgency → patient_summary → END

Nodes in graph: ['knowledge_agent', 'reasoning_agent', 'shap_explainer', 'critique_agent', 'lime_explainer', 'urgency_classifier', 'patient_summary']


PartitionExplainer explainer: 2it [00:10, 10.56s/it]               


\ Urgency: EMERGENCY
  Summary: 1. We think you might be having a heart problem, possibly a condition where the heart muscle is not getting enough blood, which could be causing your chest pain, left arm numbness, and shortness of breath.
2. We think this because you are experiencing a combination of symptoms that are often associated with heart problems, including chest pain and shortness of breath, and the left arm numbness is a specific symptom that can occur when the heart is not functioning properly.
3. You should see a doctor right away, as your symptoms could indicate a life-threatening condition that requires immediate medical attention to prevent serious harm or death, and the doctor will likely order some tests, such as an electrocardiogram and imaging tests, to determine the cause of your symptoms.

 Full pipeline working — LIME, urgency, and patient summary all functioning!


In [ ]:
# Urgency + Summary
!pip install gradio -q
import gradio as gr

def build_patient_case(age, gender, duration, symptoms):
    if not symptoms or len(symptoms.strip()) < 5:
        return None, "⚠️ Please describe the main symptoms in some detail."
    parts = []
    if age:
        parts.append(f"{age} year old")
    if gender and gender != "Prefer not to say":
        parts.append(gender.lower())
    demographic = " ".join(parts)
    duration_text = f", symptoms present for {duration}" if duration else ""
    case_text = f"{demographic} patient with {symptoms.strip()}{duration_text}".strip()
    if case_text.startswith("patient"):
        case_text = case_text[0].upper() + case_text[1:]
    return case_text, None


def run_mediagent(age, gender, duration, symptoms):
    case_text, error = build_patient_case(age, gender, duration, symptoms)
    if error:
        return error, "", "", ""

    initial_state = {
        "patient_case": case_text, "retrieved_docs": [], "reasoning_output": "",
        "shap_explanation": [], "critique_output": "", "lime_explanation": [],
        "final_diagnosis": "", "final_confidence": 0.0, "recommended_tests": "",
        "patient_summary": "", "urgency_level": "", "urgency_reason": ""
    }

    try:
        result = clinical_graph_v2.invoke(initial_state)
    except Exception as e:
        return f"Error: {str(e)}", "", "", ""

    urgency_colors = {"EMERGENCY": "🔴", "URGENT": "🟠", "ROUTINE": "🟢"}
    urgency_icon = urgency_colors.get(result['urgency_level'], "⚪")

    summary_md = f"""## {urgency_icon} Urgency: {result['urgency_level']}
{result['urgency_reason']}

---

## 📝 What This Might Mean For You

{result['patient_summary']}

**Confidence in this assessment:** {result['final_confidence']*100:.0f}%

⚠️ **This is not a medical diagnosis.** Always consult a qualified clinician.

👇 *For more details, see the detailed clinical report below.*
"""

    shap_md = "### 💡 SHAP — Key Words That Influenced the Analysis\n\n"
    for token, score in result['shap_explanation'][:6]:
        bar = "🟦" * int(score * 50)
        shap_md += f"- **{token}**: {score:.4f} {bar}\n"

    lime_md = "### 🔍 LIME — Cross-Validation Check\n\n"
    for word, score in result['lime_explanation'][:6]:
        bar = "🟩" * int(abs(score) * 100)
        lime_md += f"- **{word}**: {score:.4f} {bar}\n"

    rec_md = f"""### 🔬 Full Clinical Diagnosis (Technical)
**{result['final_diagnosis']}**

### 🧪 Recommended Next Steps
{result['recommended_tests']}

---
### 🧠 Full Clinical Reasoning (Agent 2)
{result['reasoning_output']}

---
### ⚖️ Critique & Second Opinion (Agent 3)
{result['critique_output']}
"""

    return summary_md, shap_md, lime_md, rec_md


with gr.Blocks(title="MediAgent — Explainable Clinical AI") as demo:
    gr.Markdown("""
    # 🏥 MediAgent
    ### Explainable Multi-Agent AI for Clinical Decision Support

    Please fill in as much detail as possible. More detail (age, gender, how long
    symptoms have lasted, and a clear description) helps provide a more accurate result.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            with gr.Row():
                age_input = gr.Number(label="Age", minimum=0, maximum=120, precision=0)
                gender_input = gr.Dropdown(
                    label="Gender",
                    choices=["Male", "Female", "Other", "Prefer not to say"],
                    value="Prefer not to say"
                )
            duration_input = gr.Textbox(
                label="How long have you had these symptoms?",
                placeholder="e.g. 2 days, 3 hours, 1 week"
            )
            symptoms_input = gr.Textbox(
                label="Describe your main symptoms",
                placeholder="e.g. chest pain, shortness of breath, left arm numbness",
                lines=4
            )
            submit_btn = gr.Button("🔍 Analyze Case", variant="primary")

            gr.Examples(
                examples=[
                    [45, "Male", "2 hours", "chest pain, left arm numbness and shortness of breath"],
                    [30, "Female", "1 week", "sharp chest pain that improves when sitting forward"],
                    [26, "Male", "1 year", "mild lower back pain when bending"],
                    [68, "Female", "2 weeks", "shortness of breath, leg swelling and fatigue"],
                ],
                inputs=[age_input, gender_input, duration_input, symptoms_input]
            )

        with gr.Column(scale=2):
            summary_output = gr.Markdown(label="Summary")

    gr.Markdown("---\n## 🔧 Detailed Clinical Report")

    with gr.Row():
        shap_output = gr.Markdown(label="SHAP")
        lime_output = gr.Markdown(label="LIME")
    details_output = gr.Markdown(label="Full Agent Trace")

    submit_btn.click(
        fn=run_mediagent,
        inputs=[age_input, gender_input, duration_input, symptoms_input],
        outputs=[summary_output, shap_output, lime_output, details_output]
    )

print("🚀 Launching MediAgent UI with urgency + plain-language summary...")
demo.launch(share=True, debug=False)

🚀 Launching MediAgent UI with urgency + plain-language summary...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9997ef520e70115dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
